In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'project_root: {project_root}')


project_root: /data/podo/Projects/project_jhl/20260215_nsclc


In [2]:
# autoreload
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc

In [4]:
#init_run
from src.run_setup import init_run

cfg, logger = init_run(
    config_path="../configs/nsclc_v1.json",   # <- 네가 만든 파일
    logger_name="qc",
    run_name="nsclc_v1_qc",
    note="QC run with nsclc_v1.json",
)

2026-03-09 01:13:16 | INFO | qc | Loaded config: ../configs/nsclc_v1.json
2026-03-09 01:13:16 | INFO | qc | Run name: nsclc_v1_qc | note: QC run with nsclc_v1.json


In [ ]:
from src.utils_io import load_h5ad, load_10x_h5, Timer, attach_run_config_to_adata
from src.qc import run_qc, QCParams

# 1) Load data (choose one)
with Timer(logger, "Load input (.h5ad)"):
    adata = load_h5ad("../data/20260309_pilot/nsclc_n73/20260309_adata_raw.h5ad", cfg=cfg)

# Or for 10x h5:
# with Timer(logger, "Load input (10x .h5)"):
#     adata = load_10x_h5("../data/filtered_feature_bc_matrix.h5")

2026-03-09 01:13:22 | INFO | qc | [START] Load input (.h5ad)
2026-03-09 01:14:56 | INFO | qc | [END] Load input (.h5ad) | 94.25s


In [6]:
adata

AnnData object with n_obs × n_vars = 645265 × 15170
    obs: 'nGene', 'nUMI', 'cell_barcode', 'sample', 'file', 'time', 'sample_time', 'leiden', 'anno_l2', 'anno_l1', 'anno_c1', 'sample_id', 'Sample ID', 'Study ID', 'Sex M: 1  F: 2', 'Age_IO', 'Histology', 'Smoking Never: 0 Ex: 1 Current: 2', 'ECOG _PS', 'Drug', 'PD_Event', 'PFS (Days)', 'Death_Event', 'OS (Days)', 'EGFR_Mutation_Status', 'IO_Line', 'Previous_palliative_chemo', 'Previous_palliative_target', 'PD-L1_TPS'
    var: 'n_counts'

In [7]:
# 2) Attach final config snapshot into adata
attach_run_config_to_adata(
    adata,
    cfg,
    source_config="../configs/nsclc_v1.json",
    note="Config snapshot attached at QC start",
)

In [8]:

# 3) QC params (can still override thresholds here if needed)
params = QCParams(dataset_type=cfg.dataset_type, run_doublet=True)

adata = run_qc(
    adata,
    cfg=cfg,
    params=params,
    logger=logger,
    save_name="adata_qc",
)

adata

2026-03-09 01:15:54 | INFO | qc | X type: <class 'scipy.sparse._csr.csr_matrix'> | sparse=True
2026-03-09 01:15:57 | INFO | qc | [START] QC metrics
2026-03-09 01:16:11 | INFO | qc | Computed QC metrics: total_counts, n_genes_by_counts, pct_counts_mt/ribo/hb, etc.
2026-03-09 01:16:11 | INFO | qc | [END] QC metrics | 13.19s
2026-03-09 01:16:11 | INFO | qc | Using configured QC thresholds (suggest_thresholds skipped).
2026-03-09 01:16:11 | INFO | qc | [START] QC filter cells
2026-03-09 01:16:38 | INFO | qc | Filtered cells by QC: 645265 -> 578931 (removed 66334) | min_counts=2000, max_counts=8000, min_genes=None, max_genes=None, max_pct_mt=8, max_pct_ribo=None, max_pct_hb=0.2
2026-03-09 01:16:38 | INFO | qc | [END] QC filter cells | 27.51s
2026-03-09 01:16:38 | WARNING | qc | Scrublet not installed. Skipping doublet detection.
2026-03-09 01:16:38 | INFO | qc | Saving checkpoint: data/adata_qc.h5ad


AnnData object with n_obs × n_vars = 578931 × 15170
    obs: 'nGene', 'nUMI', 'cell_barcode', 'sample', 'file', 'time', 'sample_time', 'leiden', 'anno_l2', 'anno_l1', 'anno_c1', 'sample_id', 'Sample ID', 'Study ID', 'Sex M: 1  F: 2', 'Age_IO', 'Histology', 'Smoking Never: 0 Ex: 1 Current: 2', 'ECOG _PS', 'Drug', 'PD_Event', 'PFS (Days)', 'Death_Event', 'OS (Days)', 'EGFR_Mutation_Status', 'IO_Line', 'Previous_palliative_chemo', 'Previous_palliative_target', 'PD-L1_TPS', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb'
    var: 'n_counts', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'run_config'
    layers: 'counts'

In [9]:
adata.obs.head()

,nGene,nUMI,cell_barcode,sample,file,time,sample_time,leiden,anno_l2,anno_l1,...,Previous_palliative_target,PD-L1_TPS,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,total_counts_ribo,pct_counts_ribo,total_counts_hb,pct_counts_hb
CellID,,,,,,,,,,,,,,,,,,,,,
10-IO_1st_AAACCTGAGAGCCCAA-1,2319,5257.0,AAACCTGAGAGCCCAA-1,IO_SC_029,10-IO_1st,1st,IO_SC_029-1st,9,CD8_Naive,CD8,...,X,NaN,2319,5257.0,100.0,1.902226,1850.0,35.191174,1.0,0.019022
10-IO_1st_AAACCTGAGAGTACAT-1,2282,4825.0,AAACCTGAGAGTACAT-1,IO_SC_029,10-IO_1st,1st,IO_SC_029-1st,19,cDC,DC,...,X,NaN,2282,4825.0,72.0,1.492228,635.0,13.160622,0.0,0.000000
10-IO_1st_AAACCTGAGTCATGCT-1,2270,5499.0,AAACCTGAGTCATGCT-1,IO_SC_029,10-IO_1st,1st,IO_SC_029-1st,31,Platelet,Other,...,X,NaN,2270,5499.0,89.0,1.618476,712.0,12.947808,1.0,0.018185
10-IO_1st_AAACCTGAGTGGTAAT-1,2054,5345.0,AAACCTGAGTGGTAAT-1,IO_SC_029,10-IO_1st,1st,IO_SC_029-1st,4,CD4_Naive,CD4,...,X,NaN,2054,5345.0,124.0,2.319925,2078.0,38.877453,0.0,0.000000
10-IO_1st_AAACCTGCACGCCAGT-1,1780,4948.0,AAACCTGCACGCCAGT-1,IO_SC_029,10-IO_1st,1st,IO_SC_029-1st,8,CD8_TCM,CD8,...,X,NaN,1780,4948.0,163.0,3.294261,1411.0,28.516573,6.0,0.121261


In [ ]:
adata.write('../data/20260309_pilot/nsclc_n73/20260309_adata_QC.h5ad')